In [ ]:
import csv
from pathlib import Path

surfaces = []
prix = []

PRIX_MIN = 50_000
PRIX_MAX = 1_000_000
SURFACE_MIN = 10
SURFACE_MAX = 250

csv_path = Path.cwd().parent / "donnees" / "DVF-83-Toulon-2024-2025Brut.csv"
print(csv_path)

with open(csv_path, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    for row in reader:
        try:
            if row["type_local"] in ["Appartement", "Maison"] and row["nature_mutation"] == "Vente":
                s = float(row["surface_reelle_bati"].replace(",", "."))
                p = float(row["valeur_fonciere"].replace(",", "."))

                if SURFACE_MIN <= s <= SURFACE_MAX and PRIX_MIN <= p <= PRIX_MAX:
                    surfaces.append(s)
                    prix.append(p)

        except (ValueError, TypeError, KeyError, AttributeError):
            continue

print(f"{len(surfaces)} transactions qualifiées chargées.")


/Users/jeremyindelicato/Desktop/projet-1-observatoire-immobilier-tool-on/donnees/DVF-83-Toulon-2024-2025clean.csv
0 transactions qualifiées chargées.


In [10]:
def mean(xs): 
    return sum(xs) / len(xs)

def median(xs):
    sorted_xs = sorted(xs)
    n = len(xs)
    midpoint = n // 2
    if n % 2 == 1:
        return sorted_xs[midpoint]
    return (sorted_xs[midpoint - 1] + sorted_xs[midpoint]) / 2

def variance(xs):
    mu = mean(xs)
    return sum((x - mu) ** 2 for x in xs) / (len(xs) - 1)

def standard_deviation(xs):
    return math.sqrt(variance(xs))

# Calculs sur les données réelles
print(f"Prix Moyen : {mean(prix):,.2f} €")
print(f"Prix Médian : {median(prix):,.2f} €")
print(f"Écart-type : {standard_deviation(prix):,.2f} €")

Prix Moyen : 225,296.16 €
Prix Médian : 165,050.00 €
Écart-type : 165,269.56 €


In [11]:
def covariance(xs, ys):
    mu_x, mu_y = mean(xs), mean(ys)
    return sum((x - mu_x) * (y - mu_y) for x, y in zip(xs, ys)) / (len(xs) - 1)

def correlation(xs, ys):
    stdev_x, stdev_y = standard_deviation(xs), standard_deviation(ys)
    if stdev_x > 0 and stdev_y > 0:
        return covariance(xs, ys) / (stdev_x * stdev_y)
    return 0

print(f"Coefficient de corrélation : {correlation(surfaces, prix):.4f}")

Coefficient de corrélation : 0.5404


In [12]:
def least_squares_fit(x, y):
    beta = correlation(x, y) * (standard_deviation(y) / standard_deviation(x))
    alpha = mean(y) - beta * mean(x)
    return alpha, beta

def predict(alpha, beta, x_i):
    return alpha + beta * x_i

# Calcul du modèle NidDouillet
alpha, beta = least_squares_fit(surfaces, prix)

# Exemple concret pour un conseiller immobilier [cite: 14]
surface_test = 50
estimation = predict(alpha, beta, surface_test)

print(f"Modèle : Prix = {alpha:,.2f} + {beta:,.2f} * Surface")
print(f"Estimation pour {surface_test} m² : {estimation:,.2f} €")

Modèle : Prix = 34,089.71 + 3,091.28 * Surface
Estimation pour 50 m² : 188,653.53 €


In [13]:
def r_squared(alpha, beta, x, y):
    # Somme des erreurs au carré
    ss_res = sum((y_i - predict(alpha, beta, x_i))**2 for x_i, y_i in zip(x, y))
    # Variation totale
    mean_y = mean(y)
    ss_tot = sum((y_i - mean_y)**2 for y_i in y)
    return 1 - (ss_res / ss_tot)

print(f"Coefficient de détermination R² : {r_squared(alpha, beta, surfaces, prix):.4f}")

Coefficient de détermination R² : 0.2920
